Trying to create a personal 3D stitiching algorithm

In [19]:
import tifffile
import pandas as pd
from skimage.measure import regionprops, regionprops_table
import napari
import numpy as np

def properties_mask(image):

    # Get the properties of every mask in the image
    props = regionprops(image)

    data = []

    # For every mask found, get the label, centeroid, boundingbox, and volume
    for prop in props:
        label = prop.label
        centroid = prop.centroid  # (z, y, x)
        bounding_box = prop.bbox  # (min_z, min_y, min_x, max_z, max_y, max_x)
        volume = prop.area
        
        # Pre-compute coordinate set for fast IOU calculations later
        coords_2d = prop.coords[:, 1:]
        coords_set = set(map(tuple, coords_2d))

        data.append(
            {
                "label": label,
                "z": centroid[0],
                "y": centroid[1],
                "x": centroid[2],
                "bounding_box": bounding_box,
                "volume": volume,
                "coords_2d": coords_2d,
                "coords_set": coords_set,
            }
        )

    # Make a pandas data frame from the data
    df_props = pd.DataFrame(data)

    return df_props


def properties_channel(mask, image, type=None):

    # Get the properties of the mask for the channel you specified (image)
    props_channel = regionprops_table(
        mask, intensity_image=image, properties=["label", "mean_intensity"]
    )

    # Make this into a pandas dataframe and rename some variables to the name of the channel for later merging
    props_channel = pd.DataFrame(props_channel).rename(
        columns={
            "label": "label",
            "mean_intensity": f"raw_{type}",
        }
    )

    return props_channel

def make_unique_mask(mask):
    """Makes a fresh mask from a already stitched mask"""
    unique_mask = []
    number = 1
    for z in range(mask.shape[0]):
        # go over every unique number in the mask and create a new slice with a new unique number 1 higher
        slice = mask[z]
        unique_slice = mask[z].copy()
        for val in range(1, slice.max() + 1):
            unique_slice[slice == val] = number
            number += 1
        unique_mask.append(unique_slice)
    unique_mask = np.stack(unique_mask, axis=0)

    return unique_mask

Dual nuclei image

In [27]:
# generate mask
import os
import sys
sys.path.insert(0, os.path.dirname(os.getcwd()))
import utils

image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)

nuclei = image[:, 0:1]

nuclei = np.max(image, axis=1)

model = utils.load_model(r"C:\Users\6331823\Local SSD\Train dual nuclei 20x\models\dual_nuclei")

mask = utils.segment(nuclei, model)

tifffile.imwrite(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif",
    mask,
)

Using CUDA for processing.
loaded custom model: dual_nuclei


In [31]:
image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-0.tif"
)
crc_image = image[:,0]
wt_image = image[:,1]
mask = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\Frame-0_segmented.tif"
)
# print(mask.shape)
# # to the mask, add pixel on the left side with value 0 for broken image
# mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode='constant', constant_values=0)
# print(mask.shape)

unique_mask = make_unique_mask(mask)

props_mask = properties_mask(unique_mask)
props_wt = properties_channel(unique_mask, wt_image, type="wt")
props_crc = properties_channel(unique_mask, crc_image, type="crc")

# Merge all the dataframes together on the label
df_final = props_mask.merge(props_wt, on="label").merge(props_crc, on="label")

In [10]:
def calculate_iou(idx1, idx2):
    """Ultra-fast IOU using pre-computed sets with early exit"""
    set1 = coords_sets[idx1]
    set2 = coords_sets[idx2]

    # Check if either set is empty
    len1, len2 = len(set1), len(set2)
    if len2 == 0:
        return 0
    
    # Calculate intersection (overlapping pixels)
    intersection = len(set1 & set2)  # Set intersection operator
    if intersection == 0:
        return 0  # No overlap = different cells

    # Calculate union (all unique pixels from both sets)
    union = len1 + len2 - intersection
    return intersection / union

In [32]:
import time

total_start_time = time.time()

print("Initializing data structures...")
# Reset index to ensure sequential 0-based indexing
df_final = df_final.reset_index(drop=True)

df_final["3d_label"] = -1  # Use -1 to indicate unassigned
mask_3D = np.zeros_like(unique_mask)
coords_sets = df_final["coords_set"].to_dict()

z_groups = {}
for idx, z_val in df_final["z"].items():
    z_int = int(round(z_val))
    if z_int not in z_groups:
        z_groups[z_int] = []
    z_groups[z_int].append(idx)

z_array = df_final["z"].values
x_array = df_final["x"].values
y_array = df_final["y"].values
labels_array = np.full(len(df_final), -1, dtype=np.int32)

init_time = time.time() - total_start_time
print(f"Initialization completed in {init_time:.3f}s")

print("Starting main stitching process...")
stitching_start = time.time()


def stitch_forward(idx, current_label):
    """Simplified recursive stitching - idx is now both pandas index and array position"""
    current_z = int(round(z_array[idx]))

    # Assign the current label
    labels_array[idx] = current_label

    # Fast lookup: get only indices in the next slice
    next_z = current_z + 1
    if next_z not in z_groups:
        return

    candidates = z_groups[next_z]

    # Filter unassigned candidates
    unassigned_candidates = [c for c in candidates if labels_array[c] == -1]

    if len(unassigned_candidates) == 0:
        return

    # Vectorized distance calculation
    unassigned_array = np.array(unassigned_candidates)
    distances = np.sqrt(
        (x_array[unassigned_array] - x_array[idx]) ** 2
        + (y_array[unassigned_array] - y_array[idx]) ** 2
    )

    # Find the closest candidate
    closest_idx = np.argmin(distances)
    closest_distance = distances[closest_idx]

    # Only proceed if the closest cell is within threshold
    if closest_distance > 10:
        return

    # Get the closest candidate's index
    closest_candidate = unassigned_array[closest_idx]

    # Check IOU with the closest candidate only
    iou = calculate_iou(idx, closest_candidate)
    if iou > 0.3:
        # Recursively stitch this cell forward
        stitch_forward(closest_candidate, current_label)


unique_label = 0
for idx in range(len(df_final)):
    if labels_array[idx] == -1:
        unique_label += 1
        stitch_forward(idx, unique_label)

# Copy results back to dataframe
df_final["3d_label"] = labels_array

stitching_time = time.time() - stitching_start
print(f"Main stitching completed in {stitching_time:.3f}s")
print(f"Total unique 3D cells found: {unique_label}")

print("Creating final 3D mask...")
mask_creation_start = time.time()
# Vectorized mask creation
for idx, row in df_final.iterrows():
    z = int(row["z"])
    label_3d = labels_array[idx]
    mask_3D[z][unique_mask[z] == row["label"]] = label_3d

mask_creation_time = time.time() - mask_creation_start
print(f"Final 3D mask created in {mask_creation_time:.3f}s")

Initializing data structures...
Initialization completed in 0.006s
Starting main stitching process...
Main stitching completed in 0.034s
Total unique 3D cells found: 335
Creating final 3D mask...
Final 3D mask created in 0.088s


In [33]:
import napari
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(mask_3D, name="3D_stitched_mask", scale = (2.5,0.64,0.64))
viewer.add_labels(unique_mask, name="original_mask", scale = (2.5,0.64,0.64))

<Labels layer 'original_mask' at 0x246d97cbed0>